# Serve + Test the E4B adapter on Colab T4 (HF nf4 — full fidelity)

Serves your trained E4B LoRA at the SAME nf4 4-bit precision it was trained on (no GGUF
quant degradation), on a free Colab **T4 (16GB)** which fits E4B 4-bit with room. This is
the truest read of model quality — q4_0 GGUF would confound it (and we've seen q4_0
fabricate numbers).

**Two things:** Cell 7 runs my probe prompts (you read the raw behavior); Cell 8 serves the
web UI via an ngrok tunnel so YOU can chat-test in the browser.

**Before you run:** Runtime → Change runtime type → **T4 GPU**. Add Colab Secrets:
`HF_TOKEN` (HF read) and `NGROK_TOKEN` (free from ngrok.com → authtoken). Put your
downloaded `best/` adapter folder in Google Drive (e.g. `MyDrive/chess_adapter/best`).

In [ ]:
# Cell 1 - config
import os
BASE_REPO = "unsloth/gemma-4-E4B-it"   # the base your adapter was trained on
REPO_URL  = "https://github.com/RyanDev1st/llm-and-engine.git"
BRANCH    = "feat/chess-coach-sft"
WORKDIR   = "/content/llm-and-engine"
# Where your downloaded best/ adapter lives. Drive is simplest; or upload via Cell 6b.
ADAPTER_DIR = "/content/drive/MyDrive/chess_adapter/best"
BASE_DIR  = f"{WORKDIR}/src/llm/models/gemma4_e4b"
print("base:", BASE_REPO, "| adapter:", ADAPTER_DIR)

In [ ]:
# Cell 2 - GPU check (want a T4 16GB)
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.free","--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print("device:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 3 - clone repo
import os, subprocess
env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    subprocess.run(["git","clone","--depth","1","--branch",BRANCH,REPO_URL,WORKDIR], check=True, env=env)
else:
    subprocess.run(["git","-C",WORKDIR,"pull","--ff-only"], check=True, env=env)
subprocess.run(["git","-C",WORKDIR,"log","-1","--oneline"])
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("PYTHONPATH=", os.environ["PYTHONPATH"])

In [ ]:
# Cell 4 - deps (serve path + ngrok + stockfish for full chess tools)
import subprocess, sys
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=True)
pip("-U","transformers","accelerate","peft","bitsandbytes","sentencepiece","protobuf","python-chess","pyngrok")
subprocess.run(["apt-get","-qq","install","-y","stockfish"])  # chess tools (optional)
import transformers, peft, bitsandbytes
print("transformers",transformers.__version__,"| peft",peft.__version__,"| bnb",bitsandbytes.__version__)

In [ ]:
# Cell 5 - HF login + download the E4B base (~9GB; once)
import os
from huggingface_hub import login, snapshot_download
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception:
    login(os.environ["HF_TOKEN"])
snapshot_download(repo_id=BASE_REPO, local_dir=BASE_DIR,
                  allow_patterns=["*.json","*.safetensors","*.model","*.txt","tokenizer*"])
print("base at", BASE_DIR, "->", sorted(os.listdir(BASE_DIR))[:8])

In [ ]:
# Cell 6 - mount Drive + locate the adapter
import os
from google.colab import drive
drive.mount("/content/drive")
need = ("adapter_model.safetensors","adapter_config.json")
ok = all(os.path.exists(f"{ADAPTER_DIR}/{f}") for f in need)
print("adapter at", ADAPTER_DIR, "->", os.listdir(ADAPTER_DIR) if ok else "NOT FOUND")
assert ok, f"put best/ at {ADAPTER_DIR} (or fix ADAPTER_DIR / use Cell 6b to upload)"

In [ ]:
# Cell 6b - (alt) upload the adapter instead of Drive. Zip best/ locally, upload here.
# from google.colab import files; import zipfile, io, os
# up = files.upload()                       # pick best.zip
# name = next(iter(up))
# ADAPTER_DIR = "/content/best"
# with zipfile.ZipFile(io.BytesIO(up[name])) as z: z.extractall("/content")
# print(os.listdir(ADAPTER_DIR))

In [ ]:
# Cell 7 - PROBE: load base+adapter (nf4) and read the raw behavior on key scenarios.
# Builds the SAME system prompt the production serve uses (build_system) and generates ONE
# step per scenario, so you see exactly what action the model takes.
import os, sys
os.environ["CHESS_HF_BASE"] = BASE_DIR    # serve code picks the E4B base (not the e2b default)
sys.path.insert(0, f"{WORKDIR}/src/llm")
from backend.model_hf import HFModel
from backend.inference import build_system_prompt

print("loading base+adapter (nf4) ...", flush=True)
m = HFModel(adapter=ADAPTER_DIR, temperature=0.0)
print("loaded.\n")

def step(prompt, mode, stop=("</tool>","</skill>"), mx=220):
    sysp = build_system_prompt(reasoning_mode=mode)
    msgs = [{"role":"system","content":sysp},{"role":"user","content":prompt}]
    out = m.generate(msgs, mx, list(stop))
    print(f"### [{mode}] {prompt}\n{out}\n{'-'*70}")

# 1) verification-as-tool-use (V1_R): should emit <tool>python code=...> not a guessed number
step("I scored 78, 92, and 85 — is my average above 85?", "think")
step("What's a 18% tip on a $124.60 bill?", "auto")
# 2) routing: should load the fitting <skill> for the domain
step("Can you review this SQL query for bugs and tell me why it's slow?", "think")
# 3) chess routing + grounding
step("What's the best move here?", "auto")
# 4) mode toggle: FAST must NOT emit <think>
step("hey, what can you do?", "fast")
step("hey, what can you do?", "think")

In [ ]:
# Cell 8 - SERVE the web UI via ngrok so YOU can chat-test in the browser.
# Starts the persistent model service (loads weights once) + the weightless app, tunnels :7860.
import os, sys, subprocess, time
os.environ["CHESS_HF_BASE"] = BASE_DIR
env = {**os.environ, "PYTHONPATH": f"{WORKDIR}/src/llm",
       "CHESS_HF_ADAPTER": ADAPTER_DIR, "CHESS_MODEL_SERVER": "http://127.0.0.1:7861",
       "CHESS_HOST": "127.0.0.1", "CHESS_PORT": "7860"}
# 1) model service (loads E4B base + adapter once) -> :7861
msvc = subprocess.Popen([sys.executable,"-u","-m","backend.model_server", ADAPTER_DIR],
                        cwd=f"{WORKDIR}/src/llm", env=env)
print("model service starting (loads weights, ~1-2 min)...")
import urllib.request, json
for _ in range(180):
    time.sleep(2)
    try:
        h = json.load(urllib.request.urlopen("http://127.0.0.1:7861/health"))
        if h.get("ok"): print("model service UP, adapter=", h.get("adapter")); break
    except Exception: pass
else:
    raise RuntimeError("model service didn't come up — check output above")
# 2) app server (weightless) -> :7860
app = subprocess.Popen([sys.executable,"-u","-m","backend.server"], cwd=f"{WORKDIR}/src/llm", env=env)
time.sleep(4)
# 3) ngrok tunnel
from pyngrok import ngrok
try:
    from google.colab import userdata; ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
except Exception:
    ngrok.set_auth_token(os.environ["NGROK_TOKEN"])
url = ngrok.connect(7860)
print("\n==============================================")
print("  OPEN THIS IN YOUR BROWSER TO CHAT-TEST:")
print(" ", url)
print("==============================================")
print("(leave this cell running. Stop it to shut the server down.)")